# Swiggy Revenue Forecasting using Facebook Prophet

## Problem
Swiggy wants to understand historical revenue patterns and forecast future demand in order to improve operational planning, staffing allocation, and marketing strategy.

The dataset contains order-level data including restaurant details, food categories, pricing, and customer ratings across multiple cities.

## Analysis
Exploratory Data Analysis (EDA) and statistical testing were conducted to understand revenue behavior across different factors such as cities, restaurants, food categories, quarterly trends, and customer ratings.

Statistical tests including T-test, ANOVA, and correlation analysis were used to identify significant drivers of revenue.

## Model
A Facebook Prophet time-series forecasting model was implemented to capture long-term trends and seasonal patterns in revenue data.

Additionally, feature engineering was applied by introducing a weekend indicator variable (is_weekend) as an external regressor, allowing the model to capture behavioral demand patterns between weekdays and weekends.

## Result
The forecasting model successfully captured daily, weekly, and monthly revenue trends.

Results indicate that weekend demand significantly increases revenue, and incorporating this feature improved the model's ability to capture short-term demand fluctuations.

## Model Evaluation

Model performance was evaluated using cross-validation to measure forecasting accuracy.

Evaluation metrics used:

- MAE (Mean Absolute Error)
- RMSE (Root Mean Squared Error)
- MAPE (Mean Absolute Percentage Error)

These metrics help quantify the difference between predicted and actual revenue values and assess the reliability of the forecasting model.

## Business Impact
Revenue forecasting enables food delivery platforms and restaurant partners to:

- Optimize staffing and delivery logistics

- Plan targeted promotional campaigns

- Improve inventory management

- Anticipate peak demand periods

## Forecasting Approach

This project applies the Facebook Prophet time-series forecasting model to predict Swiggy revenue trends.

Prophet was selected because it effectively models trend changes, seasonality, and missing values commonly found in business time-series data.

The forecasting pipeline includes:

- Daily revenue forecasting for short-term demand  
- Weekly forecasting to analyze operational trends  
- Monthly forecasting to understand long-term revenue patterns  
- Feature engineering to incorporate weekend demand signals as an external regressor  

Cross-validation was used to evaluate model performance using MAE, RMSE, and MAPE metrics.

## Install Library

In [ ]:
# Install Prophet library used for time-series forecasting
# Prophet is a forecasting tool developed by Facebook (Meta)
# It is designed to handle seasonality, trends, and missing data well
!pip install prophet

##Import Libraries

In [ ]:
# Import pandas for data manipulation and data analysis
import pandas as pd

# Import numpy for numerical operations and calculations
import numpy as np

# Import matplotlib for data visualization and plotting
import matplotlib.pyplot as plt

# Import seaborn for advanced statistical visualizations
import seaborn as sns

# Import statistical tests from scipy
# ttest_ind is used for comparing means between two groups
from scipy.stats import ttest_ind

# f_oneway is used for ANOVA testing between multiple groups
from scipy.stats import f_oneway

# pearsonr calculates correlation between two numerical variables
from scipy.stats import pearsonr

# Import Prophet forecasting model
from prophet import Prophet

# tqdm is used to show progress bars in loops
from tqdm import tqdm

# Import evaluation metrics used to measure model accuracy
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Import Prophet cross validation tools for model evaluation
from prophet.diagnostics import cross_validation

# Import Prophet performance metrics function
from prophet.diagnostics import performance_metrics

# Import visualization tool for cross validation results
from prophet.plot import plot_cross_validation_metric

# Configure pandas to display all columns when printing the dataframe
pd.options.display.max_columns = None

# Set random seed for reproducibility so the model results remain consistent across runs
np.random.seed(42)

##Load Dataset

In [ ]:
# Define file path for the Swiggy analytics dataset
path = "/content/swiggy_analytics_data.csv"

# Load the dataset into a pandas DataFrame
df = pd.read_csv(path)

# Display the first few rows of the dataset
# This helps understand the structure of the data
df.head()

##Data Validation
   - shape
   - info
   - null values
   - duplicates
   - nunique
   - data types
   - describes
   - columns

This section ensures the data quality is good before analysis.

In [ ]:
# Print dataset size (number of rows and columns)
print("Dataset Shape (Rows, Columns):", df.shape)

In [ ]:
# Display information about columns
# Shows column names, data types, and non-null counts
df.info()

In [ ]:
# Check for missing values in each column
df.isnull().sum()

In [ ]:
# Check if duplicate rows exist in the dataset
df.duplicated().sum()

In [ ]:
# Count the number of unique values in each column
df.nunique()

In [ ]:
# Display data types of all columns
df.dtypes

In [ ]:
# Show summary statistics of numerical columns
# Includes mean, std, min, max, etc.
df.describe()

In [ ]:
# Display all column names in the dataset
df.columns

##Data Cleaning

In [ ]:
# Create a copy of the original dataset
# This ensures the raw dataset remains unchanged
df1 = df.copy()

# Define columns that are not needed for revenue forecasting
columns_to_drop = [
    'order_id',       # unique order identifier (not useful for forecasting)
    'Location',       # location information not used here
    'Category',       # category column redundant
    'Dish_Name',      # dish-level detail not needed for revenue trend
    'Rating_Count',   # rating count not used in this analysis
    'Rating_Status'   # rating status not needed
]

# Remove unnecessary columns from dataset
# errors='ignore' prevents crash if a column is missing
df1 = df1.drop(columns=columns_to_drop, errors='ignore')

# Preview cleaned dataset
df1.head()

## Dataset Overview

Understanding the dataset structure helps identify the scale of the data and key business dimensions such as cities, restaurants, food categories, and time coverage.

This overview provides important context before performing exploratory data analysis and building forecasting models.

In [ ]:
# Dataset overview statistics
# This summarizes key dataset characteristics such as number of orders,
# cities, restaurants, food categories, cuisine types, and the time range.

dataset_summary = {
    "Total Orders": df1.shape[0],
    "Cities": df1['City'].nunique(),
    "Restaurants": df1['Restaurant_Name'].nunique(),
    "Food Categories": df1['Food_Type'].nunique(),
    "Cuisine Types": df1['Cuisine_Type'].nunique(),
    "Start Date": df1['Order_Date'].min(),
    "End Date": df1['Order_Date'].max()
}

# Print dataset overview statistics in a readable format
print("Dataset Overview\n")

# Loop through the dataset_summary dictionary and print each metric
for key, value in dataset_summary.items():
  if isinstance(value, int):
    print(f"{key}: {value:,}")
  else:
    print(f"{key}: {value}")

##Feature Preparation
   - date conversion
   - revenue column

##Date Conversion

In [ ]:
# Convert Order_Date column into datetime format
# This is required for time series analysis
df1['Order_Date'] = pd.to_datetime(df1['Order_Date'], format='%Y-%m-%d')

# Sort the dataset by date to maintain chronological order
df1 = df1.sort_values('Order_Date')

# Preview dataset after date conversion
df1.head()

##Revenue Column

In [ ]:
# Create a new column called Revenue
# Revenue is equivalent to the order price (Price_INR represents revenue per order)
df1['Revenue'] = df1['Price_INR']

# Preview the revenue column values
df1['Revenue'].head()

##Exploratory Data Analysis
   - revenue distribution analysis
   - outlier detection

##Revenue Distribution Analysis

In [ ]:
# Calculate skewness of revenue distribution
# Skewness shows whether data is symmetric or skewed
print("Skewness Values:\n")
print(df1[['Revenue']].skew())

# Plot histogram of revenue distribution
plt.figure(figsize=(8,5))

# Histogram shows frequency distribution of revenue
# KDE line shows smoothed density curve
sns.histplot(df1['Revenue'], kde=True)

# Add chart title
plt.title("Distribution of Revenue")

# Label x-axis
plt.xlabel("Revenue")

# Label y-axis
plt.ylabel("Frequency")

# Display the plot
plt.show()

##Outlier Detection (Boxplot)

In [ ]:
# Create boxplot to visually detect outliers in revenue
plt.figure(figsize=(8,5))

# Boxplot highlights median, quartiles, and extreme values
sns.boxplot(x=df1['Revenue'])

# Add title to the plot
plt.title("Revenue Distribution")

# Display the boxplot
plt.show()

# Display statistical summary of revenue column
df1['Revenue'].describe()

##Statistical Analysis
   - T-test (weekday vs weekend)
   - ANOVA (food type / city)
   - correlation (rating vs revenue)

##T-Test (Weekday vs Weekend Revenue)

In [ ]:
# Separate revenue values for weekday orders
weekday = df1[df1['Day_Type']=='Weekday']['Revenue']

# Separate revenue values for weekend orders
weekend = df1[df1['Day_Type']=='Weekend']['Revenue']

# Perform independent two-sample T-test
# This checks if the average revenue between weekday and weekend is statistically different
t_stat, p_value = ttest_ind(weekday, weekend)

# Print calculated T-statistic
print("T-statistic (Weekday vs Weekend):", t_stat)

# Print p-value to determine significance
print("P-value (Weekday vs Weekend):", p_value)

# Define significance level
alpha = 0.05

# Hypothesis testing decision
if p_value < alpha:
    print("Reject H0")   # Means weekday and weekend revenue are statistically different
else:
    print("Fail to Reject H0")  # Means no significant difference

In [ ]:
# Create visualization to compare weekday vs weekend revenue
plt.figure(figsize=(10,5))

# Barplot showing average revenue by day type
sns.barplot(x='Day_Type', y='Revenue', data=df1)

# Add chart title
plt.title("Revenue Distribution by Weekday & Weekend")

# Rotate labels for better readability
plt.xticks(rotation=45)

# Display plot
plt.show()

##ANOVA Test (Food Type Revenue)

In [ ]:
# Create revenue groups based on food type
groups = [group['Revenue'].values for name, group in df1.groupby('Food_Type')]

# Perform ANOVA test
f_stat, p_value = f_oneway(*groups)

# Print F-statistic
print("F-statistic:", f_stat)

# Print p-value
print("P-value:", p_value)

# Define significance threshold
alpha = 0.05

# Hypothesis test result
if p_value < alpha:
    print("Reject H0")   # Revenue differs significantly across food types
else:
    print("Fail to Reject H0")

In [ ]:
# Calculate total revenue per food type
food_type_revenue = df1.groupby('Food_Type')['Revenue'].sum()

# Create donut chart showing revenue distribution
plt.figure(figsize=(10,10))

plt.pie(
    food_type_revenue,
    labels=food_type_revenue.index,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(width=0.3)
)

# Add title
plt.title("Food Type Distribution by Revenue")

# Add legend
plt.legend(loc='upper center')

# Display chart
plt.show()

##ANOVA Test (City Revenue Comparison)

In [ ]:
# Create revenue groups based on city
# Each city will have a list of revenue values
groups = [group['Revenue'].values for name, group in df1.groupby('City')]

# Perform ANOVA test to check if revenue differs across cities
f_stat, p_value = f_oneway(*groups)

# Print ANOVA F-statistic
print("F-statistic:", f_stat)

# Print p-value
print("P-value:", p_value)

# Define significance level
alpha = 0.05

# Hypothesis test decision
if p_value < alpha:
    print("Reject H0")   # At least one city has significantly different revenue
else:
    print("Fail to Reject H0")  # No significant difference between cities

In [ ]:
# Visualize average revenue by city
plt.figure(figsize=(15,10))

# Barplot showing mean revenue per city
sns.barplot(x='City', y='Revenue', data=df1, estimator=np.mean)

# Add title
plt.title("Revenue Distribution by City")

# Rotate city labels
plt.xticks(rotation=45)

# Show plot
plt.show()

##Correlation Analysis (Rating vs Revenue)

In [ ]:
# Calculate Pearson correlation between rating and revenue
corr, p_value = pearsonr(df1['Rating'], df1['Revenue'])

# Print correlation value
print("Correlation:", corr)

# Print p-value
print("P-value:", p_value)

# Define significance level
alpha = 0.05

# Hypothesis test
if p_value < alpha:
    print("Reject H0")  # Significant relationship exists
else:
    print("Fail to Reject H0")  # No strong relationship

##Quarterly Revenue Analysis
Analyze revenue trends across business quarters to identify seasonal demand patterns.

In [ ]:
# Group the dataset by Quarter and calculate total revenue for each quarter
quarterly_revenue = df1.groupby(['Year','Quarter'])['Revenue'].sum().reset_index()

# Display the quarterly revenue table
print(quarterly_revenue)

# Create a bar chart to visualize revenue distribution across quarters
plt.figure(figsize=(10,6))

sns.barplot(
    data=quarterly_revenue,
    x='Quarter',
    y='Revenue',
    hue='Year',
    palette='tab10'
)

# Add chart title and axis labels
plt.title("Quarterly Revenue Distribution")
plt.xlabel("Quarter")
plt.ylabel("Total Revenue (INR)")

# Display the plot
plt.show()

In [ ]:
# Line chart showing quarterly revenue trend

plt.figure(figsize=(10,6))

sns.lineplot(
    data=quarterly_revenue,
    x='Quarter',
    y='Revenue',
    hue='Year',
    palette='tab10',
    marker='o'
)

plt.title("Quarterly Revenue Trend")
plt.xlabel("Quarter")
plt.ylabel("Revenue")

plt.show()

##Aggregate Daily Revenue

In [ ]:
# Aggregate revenue by order date
# This converts transaction-level data into daily revenue
daily_sales = df1.groupby('Order_Date')['Revenue'].sum().reset_index()

## Time Series Forecasting with Prophet

##Time Series Preparation

In [ ]:
# Rename columns to match Prophet requirements
# Prophet requires columns named 'ds' (date) and 'y' (value)
prophet_df = daily_sales.rename(columns={
    'Order_Date': 'ds',
    'Revenue': 'y'
})

# Display first few rows of the prepared dataset
prophet_df.head()

##Daily Revenue Trend Visualization

In [ ]:
# Create line chart showing daily revenue trend over time
plt.figure(figsize=(12,6))

# Plot revenue over time
sns.lineplot(data=prophet_df, x='ds', y='y')

# Add chart title
plt.title("Daily Sales Trend")

# Label x-axis
plt.xlabel("Date")

# Label y-axis
plt.ylabel("Sales")

# Show chart
plt.show()

## Feature Engineering for Forecasting

##Feature Based Forecasting

In [ ]:
## Feature Based Forecasting

# Create weekend indicator feature
# Monday=0 ... Sunday=6
# Weekend = Saturday(5) and Sunday(6)

prophet_df['is_weekend'] = prophet_df['ds'].dt.dayofweek >= 5

# Convert boolean to numeric (0 or 1)
prophet_df['is_weekend'] = prophet_df['is_weekend'].astype(int)

# Preview dataset with new feature
prophet_df.head()

##Prophet Forecasting
   - train/test split
   - model training
   - cross validation
   - performance metrics

##Train Test Split

In [ ]:
# Split dataset into training and testing sets
# Last 30 days reserved as test set to evaluate forecasting accuracy
train = prophet_df[:-30]
test = prophet_df[-30:]

# Display dataset sizes
print("Train Size:", train.shape)
print("Test Size:", test.shape)

##Prophet Model Initialization

In [ ]:
# Initialize Prophet forecasting model
# yearly_seasonality captures yearly patterns
# weekly_seasonality captures weekly patterns
# changepoint_prior_scale controls model flexibility
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    changepoint_prior_scale=0.05
)

# Add weekend feature as external regressor
model.add_regressor('is_weekend')

##Train Prophet Model with Feature

In [ ]:
# Train the forecasting model using training data
model.fit(train)

##Cross Validation

In [ ]:
# Perform time series cross validation on the trained Prophet model
# This simulates forecasting at multiple points in time to evaluate model stability
df_cv = cross_validation(
    model,

    # initial training period used before making the first forecast
    initial='180 days',

    # spacing between cross-validation folds
    period='30 days',

    # forecasting horizon (how far into the future we predict)
    horizon='30 days'
)

# Display first few rows of cross-validation results
df_cv.head()

##Model Performance Metrics

In [ ]:
# Calculate model performance metrics using cross validation results
df_performance = performance_metrics(df_cv)

# Display first few rows of performance metrics
df_performance.head()

In [ ]:
# Convert horizon to days
df_performance['horizon_days'] = df_performance['horizon'].dt.days

# Create line plot to visualize RMSE across different forecasting horizons
plt.figure(figsize=(10,5))

# Line plot for RMSE
plt.plot(df_performance['horizon_days'], df_performance['rmse'], marker='o')

# Add grid for better readability
plt.grid(True)

# Add title to the plot
plt.title("Cross Validation RMSE vs Forecast Horizon")

# Label x-axis
plt.xlabel("Forecast Horizon (Days)")

# Label y-axis
plt.ylabel("RMSE")

# Display the plot
plt.show()

##Model Accuracy Summary

In [ ]:
# Print summary of model accuracy metrics
print("Cross Validation Summary")

# Average Mean Absolute Error
print("Average MAE:", round(df_performance['mae'].mean(),2))

# Average Root Mean Squared Error
print("Average RMSE:", round(df_performance['rmse'].mean(),2))

# Average Mean Absolute Percentage Error
print("Average MAPE:", round(df_performance['mape'].mean(),4))

##Create Future DataFrame for Prediction

In [ ]:
# Create future dates for forecasting
# periods=30 means forecasting the next 30 days
future = model.make_future_dataframe(periods=30)

# Create weekend feature for future dates
future['is_weekend'] = future['ds'].dt.dayofweek >= 5
future['is_weekend'] = future['is_weekend'].astype(int)

# Preview future dataframe
future.tail()

##Generate Forecast

In [ ]:
# Generate predictions using the trained Prophet model
forecast = model.predict(future)

# Preview forecast dataframe
forecast.head()

##Visualize Forecast Components

In [ ]:
# Plot forecast components including trend and seasonality
fig = model.plot_components(forecast)

# Add suptitle
plt.suptitle("Prophet Forecast Components", fontsize=14)

# Display the plot
plt.show()

##Train and Test Predictions

In [ ]:
# Extract predictions corresponding to training data
train_pred = forecast.iloc[:len(train)][['ds','yhat']]

# Extract predictions corresponding to test data
test_pred = forecast.iloc[len(train):][['ds','yhat']]

##Compare Actual vs Predicted Values

In [ ]:
# Merge training data with predictions
train_compare = train.merge(train_pred, on='ds')

# Merge test data with predictions
test_compare = test.merge(test_pred, on='ds')

##Train Error Metrics

In [ ]:
# Calculate training error using Mean Absolute Error
train_mae = mean_absolute_error(train_compare['y'], train_compare['yhat'])

# Calculate training error using RMSE
train_rmse = np.sqrt(mean_squared_error(train_compare['y'], train_compare['yhat']))

##Test Error Metrics

In [ ]:
# Calculate testing MAE
test_mae = mean_absolute_error(test_compare['y'], test_compare['yhat'])

# Calculate testing RMSE
test_rmse = np.sqrt(mean_squared_error(test_compare['y'], test_compare['yhat']))

##Error Results

In [ ]:
# Print training errors
print("Train MAE:", train_mae)
print("Train RMSE:", train_rmse)

# Print testing errors
print("Test MAE:", test_mae)
print("Test RMSE:", test_rmse)

##Forecast vs Actual Visualization

In [ ]:
# Merge with actual test values
comparison = test.merge(test_pred, on='ds')

# Plot actual vs predicted revenue
plt.figure(figsize=(12,6))

# Plot actual revenue
plt.plot(comparison['ds'], comparison['y'], marker='o', label="Actual")

# Plot predicted revenue
plt.plot(comparison['ds'], comparison['yhat'], marker='o', label="Predicted")

# Add grid lines
plt.grid(True)

# Show legend
plt.legend()

# Add title
plt.title("Test Forecast vs Actual")

# label axes
plt.xlabel("Date")
plt.ylabel("Revenue")

# Display plot
plt.show()

##Forecast Error Analysis

In [ ]:
# Plot forecast errors over time
plt.figure(figsize=(10,6))

# Calculate prediction error
comparison['error'] = comparison['y'] - comparison['yhat']

# Plot error values
plt.plot(comparison['ds'], comparison['error'])

# Add horizontal reference line at zero
plt.axhline(0, color='red', linestyle='--')

# Add title
plt.title("Daily Forecast Error Over Time")

# Label axes
plt.xlabel("Date")
plt.ylabel("Prediction Error")

# Show plot
plt.show()

##Forecast Future Daily Revenue

In [ ]:
# Create future dataframe to forecast the next 180 days
future = model.make_future_dataframe(periods=180)

# Add weekend feature to the future dataframe
future['is_weekend'] = future['ds'].dt.dayofweek >= 5
future['is_weekend'] = future['is_weekend'].astype(int)

# Generate daily revenue forecast using the trained Prophet model
daily_forecast = model.predict(future)

# Create new columns for better readability
# Revenue → predicted value
daily_forecast['Revenue'] = daily_forecast['yhat']

# Lower bound of prediction interval
daily_forecast['Revenue_Lower'] = daily_forecast['yhat_lower']

# Upper bound of prediction interval
daily_forecast['Revenue_Upper'] = daily_forecast['yhat_upper']

# Round forecast values to 2 decimal places
daily_forecast[['Revenue','Revenue_Lower','Revenue_Upper']] = \
daily_forecast[['Revenue','Revenue_Lower','Revenue_Upper']].round(2)

# Create final daily forecast table
daily_forecast_table = daily_forecast[['ds','Revenue','Revenue_Lower','Revenue_Upper']]

# Display last rows of forecast table
daily_forecast_table.tail()

##Daily Forecast Visualization

In [ ]:
# Create visualization of historical revenue and predicted revenue
plt.figure(figsize=(12,6))

# Plot historical revenue from training data
plt.plot(train['ds'], train['y'], label='Historical Revenue')

# Plot predicted future revenue
plt.plot(daily_forecast['ds'], daily_forecast['Revenue'], label='Forecast Revenue')

# Plot confidence interval showing uncertainty range
plt.fill_between(
    daily_forecast['ds'],
    daily_forecast['Revenue_Lower'],
    daily_forecast['Revenue_Upper'],
    alpha=0.2
)

# Add chart title
plt.title("Swiggy Daily Revenue Forecast (Prophet Model)")

# Label axes
plt.xlabel("Date")
plt.ylabel("Revenue")

# Show legend
plt.legend()

# Display chart
plt.show()

##Smooth Forecast Trend (7-Day Rolling Average)
Daily revenue can be noisy, so we smooth it.

In [ ]:
# Calculate 7-day rolling average to smooth the forecast
daily_forecast['rolling_7'] = daily_forecast['Revenue'].rolling(7).mean()

# Plot smoothed forecast trend
plt.figure(figsize=(12,6))

# Plot raw daily forecast
plt.plot(daily_forecast['ds'], daily_forecast['Revenue'], alpha=0.4, label='Daily Forecast')

# Plot smoothed trend line
plt.plot(daily_forecast['ds'], daily_forecast['rolling_7'], linewidth=3, label='7 Day Trend')

# Add legend
plt.legend()

# Add title
plt.title("Daily Revenue Forecast (Smoothed)")

# Show chart
plt.show()

##Forecast Summary Table

In [ ]:
"""##Forecast Summary"""

# Get historical revenue statistics
historical_avg = prophet_df['y'].mean()

# Get forecast period (future only)
future_forecast = daily_forecast[daily_forecast['ds'] > prophet_df['ds'].max()]

# Calculate forecast statistics
forecast_avg = future_forecast['Revenue'].mean()
forecast_max = future_forecast['Revenue'].max()
forecast_min = future_forecast['Revenue'].min()

# Calculate expected growth
growth_percent = ((forecast_avg - historical_avg) / historical_avg) * 100

# Create summary table
forecast_summary = pd.DataFrame({
    "Metric": [
        "Historical Average Revenue",
        "Forecast Average Revenue",
        "Forecast Maximum Revenue",
        "Forecast Minimum Revenue",
        "Expected Revenue Growth (%)"
    ],
    "Value": [
        round(historical_avg,2),
        round(forecast_avg,2),
        round(forecast_max,2),
        round(forecast_min,2),
        round(growth_percent,2)
    ]
})

# Display summary
print("Revenue Forecast Summary")
print("========================")
print(forecast_summary)

##Weekly Revenue Forecast

In [ ]:
# Aggregate revenue into weekly totals
weekly_sales = prophet_df.set_index('ds').resample('W')['y'].sum().reset_index()

In [ ]:
# Split weekly data into training and testing sets
train_week = weekly_sales[:-8]
test_week = weekly_sales[-8:]

# Display dataset sizes
print("Weekly Train Size:", train_week.shape)
print("Weekly Test Size:", test_week.shape)

In [ ]:
# Initialize Prophet model for weekly forecasting
weekly_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    changepoint_prior_scale=0.05
)

# Train weekly forecasting model
weekly_model.fit(train_week)

# Generate future weekly dates for forecasting
future_week = weekly_model.make_future_dataframe(periods=26, freq='W')

# Predict weekly revenue
weekly_forecast = weekly_model.predict(future_week)

# Rename columns for clarity
weekly_forecast['Revenue'] = weekly_forecast['yhat']
weekly_forecast['Revenue_Lower'] = weekly_forecast['yhat_lower']
weekly_forecast['Revenue_Upper'] = weekly_forecast['yhat_upper']

# Round forecast values
weekly_forecast[['Revenue','Revenue_Lower','Revenue_Upper']] = \
weekly_forecast[['Revenue','Revenue_Lower','Revenue_Upper']].round(2)

# Create weekly forecast table
weekly_forecast_table = weekly_forecast[['ds','Revenue','Revenue_Lower','Revenue_Upper']]

# Display last rows
weekly_forecast_table.tail()

##Weekly Forecast Visualization

In [ ]:
# Plot historical and predicted weekly revenue
plt.figure(figsize=(12,6))

# Plot historical weekly revenue
plt.plot(train_week['ds'], train_week['y'], label='Historical Revenue')

# Plot forecast
plt.plot(weekly_forecast['ds'], weekly_forecast['Revenue'], label='Forecast')

# Plot confidence interval
plt.fill_between(
    weekly_forecast['ds'],
    weekly_forecast['Revenue_Lower'],
    weekly_forecast['Revenue_Upper'],
    alpha=0.2
)

# Show legend
plt.legend()

# Add title
plt.title("Swiggy Weekly Revenue Forecast (Prophet Model)")

# Label axes
plt.xlabel("Week")
plt.ylabel("Revenue")

# Display chart
plt.show()

##Weekly Forecast Error Evaluation

In [ ]:
# Extract predicted values for training and testing
train_week_pred = weekly_forecast.iloc[:len(train_week)][['ds','Revenue']]
test_week_pred = weekly_forecast.iloc[len(train_week):][['ds','Revenue']]

# Merge predictions with actual values
train_week_compare = train_week.merge(train_week_pred, on='ds')
test_week_compare = test_week.merge(test_week_pred, on='ds')

# Calculate weekly forecast errors
mae = mean_absolute_error(test_week_compare['y'], test_week_compare['Revenue'])
rmse = np.sqrt(mean_squared_error(test_week_compare['y'], test_week_compare['Revenue']))

# Print weekly model accuracy
print("MAE (Weekly Forecast):", mae)
print("RMSE (Weekly Forecast):", rmse)

##Weekly Forecast Actual vs Predicted Visualization

In [ ]:
# Plot weekly forecast actual vs predicted revenue
plt.figure(figsize=(12,6))

# Plot actual revenue
plt.plot(test_week_compare['ds'], test_week_compare['y'], marker='o', label="Actual")

# Plot predicted revenue
plt.plot(test_week_compare['ds'], test_week_compare['Revenue'], marker='o', label="Predicted")

# Add grid lines
plt.grid(True)

# Show legend
plt.legend()

# Add title
plt.title("Weekly Forecast vs Actual")

# label axes
plt.xlabel("Date")
plt.ylabel("Revenue")

# Display chart
plt.show()

##Weekly Forecast Error Analysis

In [ ]:
# Calculate prediction error for weekly forecast
# Error = Actual revenue - Predicted revenue
test_week_compare['error'] = test_week_compare['y'] - test_week_compare['Revenue']

# Plot weekly prediction errors
plt.figure(figsize=(10,5))

# Plot error values over time
plt.plot(test_week_compare['ds'], test_week_compare['error'])

# Add horizontal line at zero for reference
plt.axhline(0, color='red', linestyle='--')

# Add title
plt.title("Weekly Forecast Error Over Time")

# Label axes
plt.xlabel("Date")
plt.ylabel("Prediction Error")

# Display chart
plt.show()

##Weekday vs Weekend Revenue Time Series

In [ ]:
# Aggregate revenue separately for weekday and weekend
daytype_sales = df1.groupby(['Order_Date','Day_Type'])['Revenue'].sum().reset_index()

# Extract weekday revenue data
weekday_sales = daytype_sales[daytype_sales['Day_Type']=='Weekday']

# Extract weekend revenue data
weekend_sales = daytype_sales[daytype_sales['Day_Type']=='Weekend']

# Aggregate weekday revenue by date
weekday_sales = weekday_sales.groupby('Order_Date')['Revenue'].sum().reset_index()

# Aggregate weekend revenue by date
weekend_sales = weekend_sales.groupby('Order_Date')['Revenue'].sum().reset_index()

# Rename columns to Prophet format
weekday_sales.columns = ['ds','y']
weekend_sales.columns = ['ds','y']

##Weekday Forecast Model

In [ ]:
# Initialize Prophet model for weekday revenue forecasting
weekday_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True
)

# Train weekday forecasting model
weekday_model.fit(weekday_sales)

# Create future dataframe for weekday forecast (90 days)
future_weekday = weekday_model.make_future_dataframe(periods=90)

# Generate weekday revenue forecast
weekday_forecast = weekday_model.predict(future_weekday)

##Weekday Forecast Visualization

In [ ]:
# Plot weekday revenue forecast
weekday_model.plot(weekday_forecast)

# Add title
plt.title("Weekday Revenue Forecast")

# label axes
plt.xlabel("Date")
plt.ylabel("Revenue")

# Show chart
plt.show()

##Weekend Forecast Model

In [ ]:
# Initialize Prophet model for weekend revenue forecasting
weekend_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True
)

# Train weekend forecasting model
weekend_model.fit(weekend_sales)

# Create future dataframe for weekend forecast
future_weekend = weekend_model.make_future_dataframe(periods=90)

# Generate weekend revenue forecast
weekend_forecast = weekend_model.predict(future_weekend)

##Weekend Forecast Visualization

In [ ]:
# Plot weekend revenue forecast
weekend_model.plot(weekend_forecast)

# Add title
plt.title("Weekend Revenue Forecast")

# label axes
plt.xlabel("Date")
plt.ylabel("Revenue")

# Show chart
plt.show()

##Weekday vs Weekend Forecast Visualization

In [ ]:
# Compare weekday vs weekend forecast in a single chart
plt.figure(figsize=(12,6))

# Plot weekday predicted revenue
plt.plot(weekday_forecast['ds'], weekday_forecast['yhat'], label='Weekday Forecast')

# Plot weekend predicted revenue
plt.plot(weekend_forecast['ds'], weekend_forecast['yhat'], label='Weekend Forecast')

# Show legend
plt.legend()

# Add title
plt.title("Swiggy Weekday vs Weekend Revenue Forecast (Prophet Model)")

# Label axes
plt.xlabel("Date")
plt.ylabel("Revenue")

# Display chart
plt.show()

##Monthly Revenue Forecast

In [ ]:
# Aggregate revenue into monthly totals
monthly_sales = prophet_df.set_index('ds').resample('ME')['y'].sum().reset_index()

In [ ]:
# Split monthly data into training and testing sets
train_month = monthly_sales[:-3]
test_month = monthly_sales[-3:]

# Print dataset sizes
print("Monthly Train Size:", train_month.shape)
print("Monthly Test Size:", test_month.shape)

##Monthly Prophet Model

In [ ]:
# Initialize Prophet model for monthly forecasting
monthly_model = Prophet(
    yearly_seasonality=False,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.05
)

# Train monthly forecasting model
monthly_model.fit(train_month)

# Create future dataframe for next 9 months
future_month = monthly_model.make_future_dataframe(periods=9, freq='ME')

# Generate monthly revenue forecast
monthly_forecast = monthly_model.predict(future_month)

# Rename prediction columns
monthly_forecast['Revenue'] = monthly_forecast['yhat']
monthly_forecast['Revenue_Lower'] = monthly_forecast['yhat_lower']
monthly_forecast['Revenue_Upper'] = monthly_forecast['yhat_upper']

# Round forecast values
monthly_forecast[['Revenue','Revenue_Lower','Revenue_Upper']] = \
monthly_forecast[['Revenue','Revenue_Lower','Revenue_Upper']].round(2)

# Create monthly forecast table
monthly_forecast_table = monthly_forecast[['ds','Revenue','Revenue_Lower','Revenue_Upper']]

# Display last rows
monthly_forecast_table.tail()

##Monthly Forecast Visualization

In [ ]:
# Plot monthly historical revenue and forecast
plt.figure(figsize=(12,6))

# Plot historical monthly revenue
plt.plot(train_month['ds'], train_month['y'], label='Historical Revenue')

# Plot forecast monthly revenue
plt.plot(monthly_forecast['ds'], monthly_forecast['Revenue'], label='Forecast Revenue')

# Plot confidence interval
plt.fill_between(
    monthly_forecast['ds'],
    monthly_forecast['Revenue_Lower'],
    monthly_forecast['Revenue_Upper'],
    alpha=0.2
)

# Add title
plt.title("Swiggy Monthly Revenue Forecast (Prophet Model)")

# Label axes
plt.xlabel("Month")
plt.ylabel("Revenue")

# Show legend
plt.legend()

# Display chart
plt.show()

##Monthly Forecast Error Evaluation

In [ ]:
# Extract predicted values corresponding to training period
train_month_pred = monthly_forecast.iloc[:len(train_month)][['ds','Revenue']]

# Extract predicted values corresponding to testing period
test_month_pred = monthly_forecast.iloc[len(train_month):][['ds','Revenue']]

# Merge predicted values with actual monthly revenue
train_month_compare = train_month.merge(train_month_pred, on='ds')
test_month_compare = test_month.merge(test_month_pred, on='ds')

# Calculate monthly forecast errors

# Mean Absolute Error
mae = mean_absolute_error(test_month_compare['y'], test_month_compare['Revenue'])

# Root Mean Squared Error
rmse = np.sqrt(mean_squared_error(test_month_compare['y'], test_month_compare['Revenue']))

# Print monthly model accuracy
print("MAE (Monthly Forecast):", mae)
print("RMSE (Monthly Forecast):", rmse)

##Monthly Forecast Actual vs Predicted Visualization

In [ ]:
# Plot Monthly Forecast Actual vs Predicted Visualization
plt.figure(figsize=(12,6))

# Plot actual revenue
plt.plot(test_month_compare['ds'], test_month_compare['y'], marker='o', label="Actual")

# Plot predicted revenue
plt.plot(test_month_compare['ds'], test_month_compare['Revenue'], marker='o', label="Predicted")

# Add grid lines
plt.grid(True)

# Show legend
plt.legend()

# Add title
plt.title("Monthly Forecast vs Actual")

# label axes
plt.xlabel("Date")
plt.ylabel("Revenue")

# Display chart
plt.show()

##Monthly Forecast Error Visualization

In [ ]:
# Calculate monthly prediction error
test_month_compare['error'] = test_month_compare['y'] - test_month_compare['Revenue']

# Plot monthly forecast error
plt.figure(figsize=(10,5))

plt.plot(test_month_compare['ds'], test_month_compare['error'])

# Add zero reference line
plt.axhline(0, color='red', linestyle='--')

# Add title
plt.title("Monthly Forecast Error Over Time")

# Label axes
plt.xlabel("Date")
plt.ylabel("Prediction Error")

# Display chart
plt.show()

##Export Forecast Results
   - daily forecast
   - forecast summary
   - weekly forecast
   - weekday vs weekend forecast
   - monthly forecast

Saving forecast results allows business teams to use predictions.

In [ ]:
# Export daily forecast results to CSV
daily_forecast.to_csv("daily_revenue_forecast.csv", index=False)

# Export forecast summary results to CSV
forecast_summary.to_csv("forecast_summary.csv", index=False)

# Export weekly forecast results to CSV
weekly_forecast.to_csv("weekly_revenue_forecast.csv", index=False)

# Export weekday forecast results to CSV
weekday_forecast.to_csv("weekday_revenue_forecast.csv", index=False)

# Export weekend forecast results to CSV
weekend_forecast.to_csv("weekend_revenue_forecast.csv", index=False)

# Export monthly forecast results to CSV
monthly_forecast.to_csv("monthly_revenue_forecast.csv", index=False)

##Revenue Trend Visualization

In [ ]:
# Plot overall restaurant revenue trend
plt.figure(figsize=(12,6))

# Plot revenue over time
sns.lineplot(data=df1, x='Order_Date', y='Revenue')

# Add title
plt.title("Restaurant Revenue Over Time")

# Label axes
plt.xlabel("Date")
plt.ylabel("Revenue")

# Display chart
plt.show()

##Top Restaurants by Revenue

In [ ]:
# Identify top 10 restaurants by total revenue
top_restaurants = (
    df1.groupby('Restaurant_Name')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

# Plot top revenue generating restaurants
plt.figure(figsize=(10,6))

sns.barplot(
    x=top_restaurants.values,
    y=top_restaurants.index
)

plt.title("Top 10 Restaurants by Revenue")
plt.xlabel("Total Revenue")
plt.ylabel("Restaurant")

plt.show()

## Category-Level Revenue Forecasting

##Forecasting by Category
This function automatically creates forecasts for each category such as:

*   restaurant

*   food type

*   food confidence

*   cuisine

*   category

*   price category

*   city

*   state

In [ ]:
# Creates forecasts for each category
def forecast_by_category(data, category_column, periods=90):

  # Dictionary to store forecasts
  forecasts = {}

  # Dictionary to store trained models
  models = {}

  # Extract unique category values
  categories = data[category_column].unique()

  # Loop through each category and build a separate Prophet forecasting model
  for cat in tqdm(categories):

    # Filter dataset for current category
    temp = data[data[category_column] == cat]

    # Aggregate revenue by date
    temp = temp.groupby('Order_Date')['Revenue'].sum().reset_index()

    # Skip categories with very small data
    if len(temp) < 30:
      continue

    # Rename columns for Prophet
    temp = temp.rename(columns={
        'Order_Date':'ds',
        'Revenue':'y'
    })

    # Initialize Prophet model
    model = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=0.05
    )

    # Train forecasting model
    model.fit(temp)

    # Create future dates
    future = model.make_future_dataframe(periods=periods)

    # Generate forecast
    forecast = model.predict(future)

    # Store trained model
    models[cat] = model

    # Store forecast results
    forecasts[cat] = forecast

  # Return all models and forecasts
  return models, forecasts

##Fix Weekday Labels in Prophet Plot
Prophet sometimes shows weekday labels as numbers (0–6).

This function replaces them with actual weekday names.

In [ ]:
# Function to correct weekday labels in Prophet weekly seasonality plot
def fix_week_labels(fig):

  # Access weekly seasonality subplot
  ax = fig.axes[1]

  # Set tick positions for days of week
  ax.set_xticks(range(7))

  # Replace numeric labels with weekday names
  ax.set_xticklabels([
      "Monday","Tuesday","Wednesday",
      "Thursday","Friday","Saturday","Sunday"
  ])

##Save Forecast Results Function
This function combines forecasts from multiple categories and saves them.

In [ ]:
# Function to save category forecasts into a single CSV file
def save_forecast_to_csv(forecast_dict, column_name, filename):

  # Create empty list to store forecasts
  forecasts = []

  # Loop through forecast dictionary
  for key in forecast_dict:

    # Copy forecast dataframe
    temp = forecast_dict[key].copy()

    # Add category name column
    temp[column_name] = key

    # Append forecast to list
    forecasts.append(temp)

  # Combine all forecasts into one dataframe
  final_forecast = pd.concat(forecasts)

  # Save combined forecast to CSV
  final_forecast.to_csv(filename, index=False)

  # Return final dataframe
  return final_forecast

##Restaurant Revenue Forecast

In [ ]:
# Generate forecasts for each restaurant
restaurant_models, restaurant_forecast = forecast_by_category(df1,'Restaurant_Name',90)

In [ ]:
# Identify top 5 revenue-generating restaurants
top_restaurants = df1.groupby('Restaurant_Name')['Revenue'] \
                    .sum() \
                    .sort_values(ascending=False) \
                    .head(5) \
                    .index

# Plot forecast components for top restaurants
for restaurant in top_restaurants:
  print("\n================================")
  print("Forecast for:", restaurant)

  fig = restaurant_models[restaurant].plot_components(
      restaurant_forecast[restaurant]
  )

  # Function calling fix weekday labels
  fix_week_labels(fig)

  plt.show()

In [ ]:
# Save restaurant forecasts to CSV
final_restaurant_forecast = save_forecast_to_csv(
    restaurant_forecast,
    "Restaurant_Name",
    "restaurant_revenue_forecast.csv"
)

##Food Type Forecast

In [ ]:
# Generate forecast by food type
foodtype_models, foodtype_forecast = forecast_by_category(df1,'Food_Type',90)

In [ ]:
# Plot forecasts for each food type
for foodtype in foodtype_models:
  print("\n================================")
  print("Forecast for:", foodtype)

  fig = foodtype_models[foodtype].plot_components(
      foodtype_forecast[foodtype]
  )

  fix_week_labels(fig)

  plt.show()

In [ ]:
# Save food type forecasts
final_food_forecast = save_forecast_to_csv(
    foodtype_forecast,
    "Food_Type",
    "food_revenue_forecast.csv"
)

##Food Confidence Forecast

In [ ]:
# Forecast revenue by food confidence category
food_confidence_models, food_confidence_forecast = forecast_by_category(df1,'Food_Type_Confidence',90)

In [ ]:
# Plot forecasts for each food type
for food_confidence in food_confidence_models:
  print("\n================================")
  print("Forecast for:", food_confidence)

  fig = food_confidence_models[food_confidence].plot_components(
      food_confidence_forecast[food_confidence]
  )

  fix_week_labels(fig)

  plt.show()

In [ ]:
# Save food type forecasts
final_food_confidence_forecast = save_forecast_to_csv(
    food_confidence_forecast,
    "Food_Type_Confidence",
    "food_confidence_revenue_forecast.csv"
)

##Cuisine Forecast

In [ ]:
# Forecast revenue by cuisine type
cuisine_models, cuisine_forecast = forecast_by_category(df1,'Cuisine_Type',90)

In [ ]:
# Plot forecasts for each food type
for cuisine in cuisine_models:
  print("\n================================")
  print("Forecast for:", cuisine)

  fig = cuisine_models[cuisine].plot_components(
      cuisine_forecast[cuisine]
  )

  fix_week_labels(fig)

  plt.show()

In [ ]:
# Save food type forecasts
final_cuisine_forecast = save_forecast_to_csv(
    cuisine_forecast,
    "Cuisine_Type",
    "cuisine_revenue_forecast.csv"
)

##Category Type Forecast

In [ ]:
# Forecast revenue by category type
category_models, category_forecast = forecast_by_category(df1,'Category_Type',90)

In [ ]:
# Plot forecasts for each food type
for category in category_models:
  print("\n================================")
  print("Forecast for:", category)

  fig = category_models[category].plot_components(
      category_forecast[category]
  )

  fix_week_labels(fig)

  plt.show()

In [ ]:
# Save food type forecasts
final_category_forecast = save_forecast_to_csv(
    category_forecast,
    "Category_Type",
    "category_revenue_forecast.csv"
)

##Price Category Forecast

In [ ]:
# Forecast revenue by price category
price_category_models, price_category_forecast = forecast_by_category(df1,'Price_Category',90)

In [ ]:
# Plot forecasts for each food type
for price_category in price_category_models:
  print("\n================================")
  print("Forecast for:", price_category)

  fig = price_category_models[price_category].plot_components(
      price_category_forecast[price_category]
  )

  fix_week_labels(fig)

  plt.show()

In [ ]:
# Save food type forecasts
final_price_category_forecast = save_forecast_to_csv(
    price_category_forecast,
    "Price_Category",
    "price_category_revenue_forecast.csv"
)

##City Forecast

In [ ]:
# Forecast revenue by city
city_models, city_forecast = forecast_by_category(df1,'City',90)

In [ ]:
# Plot forecasts for each food type
for city in city_models:

  print("\n================================")
  print("Forecast for:", city)

  fig = city_models[city].plot_components(
      city_forecast[city]
  )

  fix_week_labels(fig)

  plt.show()

In [ ]:
# Save food type forecasts
final_city_forecast = save_forecast_to_csv(
    city_forecast,
    "City",
    "city_revenue_forecast.csv"
)

##State Forecast

In [ ]:
# Forecast revenue by state
state_models, state_forecast = forecast_by_category(df1,'State',90)

In [ ]:
# Plot forecasts for each food type
for state in state_models:

  print("\n================================")
  print("Forecast for:", state)

  fig = state_models[state].plot_components(
      state_forecast[state]
  )

  fix_week_labels(fig)

  plt.show()

In [ ]:
# Save food type forecasts
final_state_forecast = save_forecast_to_csv(
    state_forecast,
    "State",
    "state_revenue_forecast.csv"
)

##Revenue Growth Forecast

In [ ]:
# Extract forecasted revenue
forecast_growth = daily_forecast[['ds','Revenue']].copy()

# Calculate percentage revenue growth compared to previous period
forecast_growth['growth_rate'] = forecast_growth['Revenue'].pct_change()*100

# Preview forecast growth data
forecast_growth.tail()

In [ ]:
# Plot revenue growth trend
plt.figure(figsize=(12,6))

plt.plot(forecast_growth['ds'], forecast_growth['growth_rate'])

plt.title("Revenue Growth Forecast")
plt.ylabel("Revenue Growth (%)")
plt.xlabel("Date")

plt.show()

In [ ]:
# Save growth forecast
forecast_growth.to_csv("forecast_growth.csv", index=False)

##Business Insights Section

In [ ]:
# Business Insights Title
print("Business Insights from Swiggy Revenue Forecasting")
print("================================================")

# Weekday vs Weekend Analysis
print("\n1. Weekday vs Weekend Demand")

weekday_avg = weekday.mean()
weekend_avg = weekend.mean()

print("Average Weekday Revenue:", round(weekday_avg,2))
print("Average Weekend Revenue:", round(weekend_avg,2))

if weekend_avg > weekday_avg:
  print("Weekend demand is significantly higher, and the forecasting model explicitly captures this pattern using a weekend feature (external regressor).")
else:
  print("Incorporating this behavioral signal improves the model’s ability to predict short-term demand fluctuations.")

# Quarterly Revenue Insight
print("\n2. Quarterly Revenue Pattern")

# Calculate revenue for each year and quarter
quarter_revenue = df1.groupby(['Year','Quarter'])['Revenue'].sum().reset_index()

# Identify the quarter with the highest revenue overall
top_quarter = (
    quarter_revenue.groupby('Quarter')['Revenue']
    .sum()
    .idxmax()
)

print("Highest revenue generating quarter:", top_quarter)

print("This indicates seasonal demand variations across different business quarters.")
print("Businesses can align marketing campaigns and delivery capacity with high-performing quarters.")

# Top City Revenue
print("\n3. City Revenue Contribution")

city_revenue = df1.groupby('City')['Revenue'].sum().sort_values(ascending=False)

top_city = city_revenue.index[0]

print("Top revenue generating city:", top_city)

print("This indicates that food delivery demand is concentrated in high-population or high-income cities.")

# Food Type Popularity
print("\n4. Food Category Demand")

food_revenue = df1.groupby('Food_Type')['Revenue'].sum().sort_values(ascending=False)

top_food = food_revenue.index[0]

print("Most popular food category:", top_food)

print("Restaurants offering this category may experience higher demand and should prioritize availability.")

# Restaurant Performance
print("\n5. Restaurant Revenue Leaders")

restaurant_revenue = df1.groupby('Restaurant_Name')['Revenue'].sum().sort_values(ascending=False)

top_restaurant = restaurant_revenue.index[0]

print("Top revenue-generating restaurant:", top_restaurant)

print("High-performing restaurants may benefit from expanded delivery coverage and marketing promotions.")

# Rating Impact
print("\n6. Rating Impact on Revenue")

print("Correlation between rating and revenue:", round(corr,2))

if abs(corr) < 0.3:
  print("Customer ratings show weak influence on revenue, suggesting other factors such as price or cuisine type drive demand.")
else:
  print("Higher rated restaurants tend to generate more revenue.")

# Forecast Growth Insight
print("\n7. Revenue Growth Forecast")

future_growth = forecast_growth['growth_rate'].mean()

print("Average predicted growth rate:", round(future_growth,2), "%")

if future_growth > 0:
  print("Revenue is expected to grow over time, indicating expanding food delivery demand.")
else:
  print("Revenue growth appears stagnant, suggesting potential market saturation.")

# Strategic Recommendation
print("\n8. Strategic Business Recommendations")

print("• Focus marketing campaigns on high-performing cities.")
print("• Promote top-selling food categories to maximize revenue.")
print("• Encourage restaurants to maintain strong ratings and customer satisfaction.")
print("• Expand delivery services during weekends when demand increases.")
print("• Use forecasting insights for inventory and staffing planning.")

## Conclusion

This project demonstrates how time-series forecasting can be used to predict food delivery demand and support data-driven decision making.

The Facebook Prophet model successfully captures long-term trends and seasonal patterns in Swiggy revenue data while incorporating behavioral signals such as weekend demand.

Key insights reveal stronger demand during weekends, higher revenue concentration in major cities, and significant contribution from specific food categories.

These insights enable food delivery platforms to optimize delivery logistics, plan promotional campaigns, allocate resources efficiently, and anticipate peak demand periods.

## Limitations and Future Work

While the Prophet model captures major seasonal patterns, forecasting accuracy could be further improved by incorporating additional features such as:

- Festival and holiday effects
- Promotional campaign data
- Weather conditions
- Restaurant supply constraints

Future work may also explore advanced forecasting models such as XGBoost, LSTM, or hybrid machine learning approaches.